# Cellformer on GPU — the AlphaFold-shaped cell model, and the ablation table that judges it

**What this runs.** An AlphaFold-shaped transformer for Perturb-seq response prediction, plus the
eleven-row ablation table specified in `CELLFORMER.md`. On the CPU sandbox where this was developed a
single training run took ~35 minutes and the full table ~6 hours, which forced crop sizes far below the
design. On a Colab GPU the crop can be **256 genes — AlphaFold's own training crop** — and the whole
table fits in well under an hour.

**The bar it has to clear.** A frequency baseline (name the globally most-moved genes, no biology at
all) scores **recall 0.2824** on the held-out set. That baseline has already beaten seven mechanistic
methods in this project. The best CPU result so far is **0.0508** — 5.5x *worse* than frequency.

**The pre-registered criterion, set before any training and unchanged here:** the full model must beat
frequency on held-out perturbations with a paired-bootstrap CI excluding zero. And if it beats frequency
while ablations 1, 3, 5 and 6 cost nothing, the honest report is *"a transformer with capacity beat the
baseline; none of its biology mattered"* — in those words.

**Run order:** cells 1–4 set up, **cell 5 is a smoke test that must pass**, then 6 (learning curve) and
7 (full table). Cell 6 answers the question that matters most right now: is the model *undertrained*, or
is the architecture simply not learning this task?

## 1. GPU check

Run this first. If it prints `cpu`, use Runtime → Change runtime type → GPU.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'no nvidia-smi')
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB, capability {p.major}.{p.minor}')
else:
    print('WARNING: no GPU. This notebook will run but slowly -- use Runtime > Change runtime type.')

## 2. Clone the repository

If the repository is private, an unauthenticated clone returns `remote: Not Found` (GitHub hides
private repos from anonymous requests rather than saying 'forbidden'). The cell detects that and asks
for a token; a fine-grained PAT with **Contents: read** on this repo is enough.

In [ ]:
REPO = 'nikku03/cell'
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'

import os, shutil, subprocess

def clone(token=None):
    if os.path.isdir('/content/cell'):
        shutil.rmtree('/content/cell')
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, '/content/cell'],
                       capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr).replace(token or '\0', '***')

rc, out = clone()
print(out)
if rc != 0:
    print('Clone failed. If this repo is private, paste a GitHub token below; press Enter to skip.')
    from getpass import getpass
    tok = getpass('GitHub token (hidden): ').strip()
    if tok:
        rc, out = clone(tok)
        print(out)
if rc != 0:
    raise SystemExit('Clone failed. Check the repo name and branch, or open this notebook via '
                     'Colab > File > Open notebook > GitHub, ticking Include private repos.')
os.chdir('/content/cell')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

## 3. Data

**The network now ships in the repo** as `colab/data/net_bundle.json.gz` (3.7 MB), so only ONE file
has to come from Drive:

| file | what it is | where |
|---|---|---|
| network bundle | reactions, complexes, PPI, co-expression, signed regulation | in the repo — nothing to do |
| `nlz_K562_gwps.npz` | the readout: 5,120 perturbations x 8,246 genes | Drive, **or** rebuilt by cell 3b |

Why the bundle is shipped rather than read from Drive: the `cell_complete.json` on Drive is 26.4 MB
against the 37.2 MB build these results came from — a different, older version. Loading it would change
the pair prior with no visible symptom, which is exactly the class of silent failure this project has
hit seven times. The bundle is the exact network the measurements used, and `load_all()` asserts every
field is populated.

**Checked on this Drive already:** `nlz_K562_gwps.npz` is NOT there, but `perturbseq_gwps_bulk.h5ad`
IS, at `MyDrive/virtual_cell_data/human_raw/` (374,587,922 bytes — byte-identical in size to the
`gwps.h5ad` these results were computed on). So run **cell 3b**.

Cell 3 indexes Drive in **one** pass and prints every candidate it found. If the search still comes up
empty, set `GWPS_H5AD` at the top of cell 3 to the full path and re-run — that skips searching
entirely. Nothing is copied: cell 3b reads the 375 MB file in place via `CELL_GWPS`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import fnmatch, os, shutil          # shutil was missing and IS used below -- NameError the moment
SCRATCH = '/content/data'           # a file was actually found, i.e. only on the success path
os.makedirs(SCRATCH, exist_ok=True)
os.makedirs('/content/cell/outputs/orphan', exist_ok=True)

# SET THIS if you know where the readout is -- it skips the search entirely.
GWPS_H5AD = ''   # e.g. '/content/drive/MyDrive/virtual_cell_data/human_raw/perturbseq_gwps_bulk.h5ad'

DRIVE = '/content/drive/MyDrive'
PATTERNS = ('nlz_K562_gwps.npz', '*gwps*.h5ad', '*gwps*.npz')
# ONE indexed pass. The previous version ran a SEPARATE recursive glob per pattern -- five full
# traversals of a FUSE-mounted Drive, minutes each. That is why the search printed 'found readout
# source: .../perturbseq_gwps_bulk.h5ad' on one run and 'not found on Drive either' on the next:
# the file never moved, the traversal did not finish the same way twice. Cheap lookups now answer
# from the index instead of re-walking.
INDEX = {}
def _add(p):
    if p and os.path.exists(p):
        INDEX.setdefault(os.path.basename(p), []).append(p)

_add(GWPS_H5AD)
for k in ('/content/drive/MyDrive/virtual_cell_data/human_raw/perturbseq_gwps_bulk.h5ad',
          '/content/drive/MyDrive/cell_model/nlz_K562_gwps.npz'):
    _add(k)                          # known locations checked first: instant, no traversal

if not any(fnmatch.fnmatch(n, '*gwps*.h5ad') for n in INDEX):
    print('indexing Drive (one pass, may take a minute) ...')
    ndir = 0
    for root, dirs, files in os.walk(DRIVE):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        ndir += 1
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in PATTERNS):
                INDEX.setdefault(fn, []).append(os.path.join(root, fn))
    print(f'  walked {ndir} directories')

print('candidates found on Drive:' if INDEX else 'no gwps/readout files found anywhere on Drive')
for n, ps in sorted(INDEX.items()):
    for p in ps:
        print(f'  {n}  {os.path.getsize(p)/1e6:.0f} MB  {p}')

def find(pattern):
    hits = [p for n, ps in INDEX.items() if fnmatch.fnmatch(n, pattern) for p in ps]
    return sorted(hits, key=os.path.getsize, reverse=True)[0] if hits else None

NEED = {'nlz_K562_gwps.npz': f'{SCRATCH}/nlz_K562_gwps.npz'}   # network ships in the repo
missing = []
for name, dest in NEED.items():
    if os.path.exists(dest) and os.path.getsize(dest) > 1e6:
        print(f'OK  {name}: already at {dest} ({os.path.getsize(dest)/1e6:.0f} MB)')
        continue
    src = find(name)
    if src:
        print(f'copying {name} from {src} ({os.path.getsize(src)/1e6:.0f} MB) ...')
        shutil.copy(src, dest)
        print(f'OK  {name} -> {dest}')
    else:
        missing.append(name)
        print(f'MISSING {name} -- not found under MyDrive/cell_model or MyDrive')

if missing:
    print('\n' + '='*70)
    print('Cannot proceed without:', ', '.join(missing))
    print('Upload them to Google Drive under MyDrive/cell_model/ and re-run this cell.')
    print('Run cell 3b to rebuild it from the h5ad -- perturbseq_gwps_bulk.h5ad is already on this Drive.')
    print('='*70)
else:
    print('\nAll data present.')

### 3b. (only if needed) rebuild the readout from `gwps.h5ad`

Skip this if cell 3 reported all data present. This reproduces `nlz_K562_gwps.npz` exactly as the
project built it — dropping the 585 non-targeting controls, averaging duplicate columns, and asserting
the matrix is finite (an earlier build silently carried 6,881 infinities that passed the `|z|>=1` mover
test, because `inf >= 1.0` is `True`).

In [ ]:
import os, subprocess
# Answered from cell 3's index -- no second traversal of Drive. The file is named
# perturbseq_gwps_bulk.h5ad here, not gwps.h5ad, so match on the pattern rather than a literal name.
src = find('*gwps*.h5ad')
if not src:
    print('no *gwps*.h5ad in the index. Set GWPS_H5AD at the top of cell 3 to its full path,')
    print('re-run cell 3, then re-run this cell. Cell 3 prints every candidate it indexed.')
else:
    print(f'using readout source: {src} ({os.path.getsize(src)/1e6:.0f} MB)')
    # DO NOT install anndata. gwps_rebuild.py imports h5py and numpy and nothing else -- anndata was
    # never needed, and installing it resolved a different numpy on top of the one already imported.
    # The session then had new numpy .py files against the old compiled core, which surfaces much
    # later as: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'.
    # h5py ships with Colab, so normally nothing is installed and numpy is never touched.
    try:
        import h5py                      # noqa: F401
        _installed = False
    except ImportError:
        subprocess.run(['pip', '-q', 'install', 'h5py'])
        _installed = True
    env = dict(os.environ, CELL_SCRATCH=SCRATCH, CELL_OUT='/content/cell/outputs/orphan',
               CELL_GWPS=src)   # read the 375 MB source IN PLACE -- no copy into scratch
    # Earlier this copied src to f'{SCRATCH}/gwps.h5ad' because gwps_rebuild.py had the sandbox path
    # hardcoded and ignored CELL_SCRATCH entirely -- so it opened /tmp/claude-0/... and raised
    # FileNotFoundError naming a path this notebook never chose. The script now reads CELL_GWPS,
    # which also removes a multi-minute duplicate of the largest file in the project.
    r = subprocess.run(['python', '-u', 'colab/gwps_rebuild.py'], cwd='/content/cell',
                       capture_output=True, text=True, env=env)
    print(r.stdout[-3000:], r.stderr[-2000:])
    if _installed:
        print('\n' + '='*70)
        print('A package was installed, which can resolve a different numpy under the one already')
        print('imported. RESTART NOW: Runtime > Restart session, then re-run cells 1-3 and skip 3b.')
        print('The rebuilt .npz is on disk and survives a restart, so nothing is recomputed.')
        print('='*70)

## 4. Point the code at the data

Every script in `colab/` reads `CELL_SCRATCH` and `CELL_OUT`, so nothing is hardcoded to the machine
this was developed on. That was not true until the `gwps_rebuild.py` failure exposed it: 86 scripts
hardcoded the sandbox scratch path and 2 read the environment, so this notebook could set the
variables correctly and still be ignored. All 279 now honour them, with the old paths as defaults.

In [ ]:
import os, sys
os.environ['CELL_SCRATCH'] = SCRATCH
os.environ['CELL_OUT'] = '/content/cell/outputs/orphan'
os.environ['CF_DEVICE'] = 'auto'
sys.path.insert(0, '/content/cell/colab')

# numpy consistency FIRST. A half-upgraded numpy -- new .py files against the compiled core that was
# already imported -- does not fail on `import numpy`. It fails nine frames deep inside whatever
# imports numpy.testing later, as:
#   AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'
# which reads like a bug in this project and is not one. Surface it here, with the actual remedy.
import numpy as np
try:
    import numpy.testing            # noqa: F401  -- the import that trips on a mismatched core
except AttributeError as e:
    raise SystemExit(
        f'numpy is inconsistent: {e}\n'
        f'  numpy {np.__version__} at {np.__file__}\n'
        '  A pip install resolved a different numpy under the one already imported.\n'
        '  FIX: Runtime > Restart session, then re-run from cell 1. Do NOT re-run any pip install.\n'
        '  Nothing is lost -- the rebuilt readout is on disk and survives the restart.')

import torch
assert os.path.exists(f'{SCRATCH}/nlz_K562_gwps.npz'), 'readout missing -- see cell 3'
assert os.path.exists('/content/cell/colab/data/net_bundle.json.gz'), 'repo bundle missing -- re-clone'
print(f'data OK; numpy {np.__version__}, torch {torch.__version__}')

## 5. SMOKE TEST — must pass before anything long

A tiny end-to-end run: load the data, build the model, train four steps, evaluate on two held-out
perturbations. It checks the things that have actually broken in this project — empty joins, a
non-finite matrix, tensors stranded on the wrong device — and it takes about a minute.

Note the assertions on the **pair channels**. The reaction channel silently produced 10 garbage pairs
once, because `generxn` maps gene index to reaction *equation strings* and the loop was parsing the
first two characters of the dict key. That channel is what the physics ablation runs on.

In [ ]:
import importlib, time
import cellformer_af as CF
importlib.reload(CF)
import torch.nn as nn

t0 = time.time()
dev = CF.set_device()
print('device:', dev)

d = CF.load_all()
assert np.isfinite(d['M']).all(), 'readout contains non-finite values'
assert len(d['pair']) > 5000, f"pair prior too sparse: {len(d['pair'])}"
assert d['pairmat'][0].nnz > 500, f"REACTION channel nearly empty: {d['pairmat'][0].nnz} nz"
assert d['pairmat'][2].nnz > 10000, f"PPI channel nearly empty: {d['pairmat'][2].nnz} nz"
print(f"joins OK -- pair channels nz: {[int(p.nnz) for p in d['pairmat']]}")

import hashlib
tr = np.array([int(hashlib.md5(k.encode()).hexdigest(), 16) % 2 == 0 for k in d['kos']])
assert 0.4 < tr.mean() < 0.6, f'split is not a half: {tr.mean():.3f}'
pool = [i for i in range(len(d['kos'])) if tr[i]]
test = [i for i in range(len(d['kos'])) if not tr[i] and d['nspec'][i] >= CF.MIN_SPEC]
print(f'split OK -- {len(pool):,} train / {len(test):,} held-out scorable')

cfg = dict(CF.BASE)
model = CF.build_model(torch, nn, len(d['genes']), cfg).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=CF.LR)
bce = nn.BCEWithLogitsLoss()
rng = np.random.default_rng(0)
losses = []
for step in range(4):
    qi = int(rng.choice(pool))
    rows, cols, qg = CF.make_example(d, rng, qi, pool, cfg, len(d['genes']))
    zin, mask, isq, gidx, koidx, pz, B, idx = CF.tensors(d, rows, cols, qg, torch, cfg, rng, False)
    for nm, t in (('zin', zin), ('pz', pz), ('gidx', gidx)):
        assert t.device.type == dev.type, f'{nm} on {t.device}, expected {dev}'
    y = torch.tensor(d['spec'][np.ix_(rows, cols)].astype(np.float32), device=dev)
    r, c, p, cap = model(zin, mask, isq, gidx, koidx, pz, B, cfg['recycle'])
    loss = bce(r[0], y[0])
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(float(loss.detach()))
assert all(np.isfinite(losses)), f'non-finite loss: {losses}'
print(f'training OK -- 4 steps, losses {[round(x,4) for x in losses]}')

freq = (np.abs(d['M'][tr]) >= CF.TAU).mean(0)
must = set()
for qi in test[:2]:
    must |= set(np.where(d['spec'][qi])[0].tolist())
ranked = [int(i) for i in np.argsort(-freq) if not d['tide'][i]]
d['cand'] = np.array(sorted(list(dict.fromkeys(sorted(must) + ranked))[:2048]), dtype=np.int64)
assert must <= set(d['cand'].tolist()), 'candidate pool drops true movers'
model.eval()
with torch.no_grad():
    qi = test[0]
    cols = list(d['cand'][:CF.G_EVAL])
    rows, _, qg = CF.make_example(d, rng, qi, pool, cfg, len(d['genes']))
    rows = rows[:1 + CF.P_CTX_EVAL]
    zin, mask, isq, gidx, koidx, pz, B, idx = CF.tensors(d, rows, cols, qg, torch, cfg, rng, False)
    r, _, _, _ = model(zin, mask, isq, gidx, koidx, pz, B, cfg['recycle'])
    sc = r[0].detach().cpu().numpy()
assert np.isfinite(sc).all() and sc.shape[0] == len(cols), 'bad eval output'
print(f'evaluation OK -- scored {len(cols)} genes')
print(f'\nSMOKE TEST PASSED in {time.time()-t0:.0f}s on {dev}')

## 6. Learning curve — undertrained, or architecture?

This is the experiment that matters most. The CPU run reached 0.0508 against a 0.2824 baseline at 150
steps, and an eleven-row table of models in that state would measure noise, not architecture. Training
**one** model and evaluating it at increasing budgets separates two conclusions the table cannot:

- recall **climbing toward 0.2824** → undertrained; run the table at the larger budget
- recall **flat near 0.05** → the architecture is not learning this task, and the table can say so

On a GPU this is minutes, not hours.

In [ ]:
os.environ['CF_GCROP'] = '256'      # AlphaFold's own training crop; the CPU run was limited to 96
os.environ['CF_PCTX'] = '128'
os.environ['CF_CM'] = '64'
os.environ['CF_CZ'] = '32'
os.environ['CF_BLOCKS'] = '4'
os.environ['CF_RECYCLE'] = '3'
os.environ['CF_GEVAL'] = '512'
os.environ['CF_PCTXEVAL'] = '16'
os.environ['CF_CKPT'] = '200,600,1500,3000'
os.environ['CF_NEVAL'] = '40'

r = subprocess.run(['python', '-u', 'colab/cellformer_curve.py'], cwd='/content/cell',
                   capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-2000:] if r.returncode else '')

## 7. The full eleven-row ablation table

Run this **after** reading the curve. Set `CF_STEPS` to a budget the curve showed is enough; if the
curve was flat, run it anyway and report the flat result honestly — that is a real finding, and it is
the case the pre-registered interpretation rule was written for.

In [ ]:
os.environ['CF_STEPS'] = '1500'     # set from what the curve showed
os.environ['CF_NEVAL'] = '40'
os.environ['CF_POOL'] = '2048'

r = subprocess.run(['python', '-u', 'colab/cellformer_af.py'], cwd='/content/cell',
                   capture_output=True, text=True)
print(r.stdout[-9000:])
print(r.stderr[-2000:] if r.returncode else '')

## 7b. Back up the full network to Drive

The repo carries `cell_complete.json.gz` (6.7 MB, the full 37.2 MB build these results came from).
This cell decompresses it and writes it to Drive, so the newer build is preserved there alongside the
older 26.4 MB copy rather than replacing it — different name, nothing overwritten.

In [ ]:
import gzip, shutil, os, json
src = '/content/cell/colab/data/cell_complete.json.gz'
dest_dir = '/content/drive/MyDrive/cell_model'
os.makedirs(dest_dir, exist_ok=True)
dest = f'{dest_dir}/cell_complete_full_37MB.json'

with gzip.open(src, 'rt') as f:
    D = json.load(f)                      # parse it, so a corrupt copy fails here and not later
n_genes = len(D['genes'])
n_ppi = len(D.get('ppi', []))
n_reg = len(D.get('reg', []))
assert n_genes > 15000 and n_ppi > 100000, 'decompressed network looks truncated'
print(f'decompressed OK -- {len(D)} top-level keys, {n_genes:,} genes, '
      f'{n_ppi:,} PPI edges, {n_reg:,} regulatory edges')
with open(dest, 'w') as f:
    json.dump(D, f)
print(f'written to Drive: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)')

# also keep the compact bundle next to it
shutil.copy('/content/cell/colab/data/net_bundle.json.gz', f'{dest_dir}/net_bundle.json.gz')
print('written to Drive: net_bundle.json.gz (3.7 MB)')

## 8. Results

In [ ]:
import json, os
for f in ('cellformer_curve.json', 'cellformer_af.json'):
    p = f'/content/cell/outputs/orphan/{f}'
    if not os.path.exists(p):
        print(f'{f}: not produced yet'); continue
    r = json.load(open(p))
    print(f'\n=== {f} ===')
    if 'curve' in r:
        print(f"frequency baseline: {r['frequency']:.4f}")
        for c in r['curve']:
            print(f"  step {c['step']:>5d}  loss {c['loss']:.4f}  recall {c['recall']:.4f}  "
                  f"({100*c['recall']/r['frequency']:.1f}% of baseline)")
    if 'results' in r:
        for a in r['results']:
            ci = a.get('ci')
            tail = f"  vs freq {a.get('delta_vs_frequency',0):+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]" if ci else ''
            print(f"  {a['tag']:<32s} recall {a['recall']:.4f}  prec {a['precision']:.4f}{tail}")

# save results back to Drive so they survive the runtime
import shutil
os.makedirs('/content/drive/MyDrive/cell_model/results', exist_ok=True)
for f in ('cellformer_curve.json', 'cellformer_af.json'):
    p = f'/content/cell/outputs/orphan/{f}'
    if os.path.exists(p):
        shutil.copy(p, f'/content/drive/MyDrive/cell_model/results/{f}')
        print('saved to Drive:', f)

## How to read the result

The number to look at is **not** the full model's recall on its own. It is:

1. **full model vs frequency (0.2824)** — with the paired-bootstrap CI. If the CI includes zero, the
   model has not beaten a baseline that uses no biology.
2. **each ablation vs the full model** — ablations 1 (no pair representation), 3 (no reaction-graph
   update), 5 (no flux module) and 6 (no physical loss) are the architecture's biological commitments.
   If removing them costs nothing, the architecture is not what produced any gain.
3. **shuffled labels (row 9)** must collapse to near zero. If it doesn't, something leaks.

Builds 3 and 4 of this project measured that the shared stress core is 37.4% of held-out response
variance and that the benchmark rewards predicting *damage* rather than *which gene was damaged*. A model
that beats frequency has to be doing the second thing, which is why the ablations matter more than the
headline number.